In [9]:
import sympy as sp
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sympy import symbols, exp, diff, simplify, pprint, latex, init_printing, solve, factorial

init_printing(use_unicode=True)

ModuleNotFoundError: No module named 'sympy'

## Leer DataFrame

In [ ]:
df = pd.read_csv('./compra_producto.csv')
# Graficar el salario en miles y la probabilidad
plt.scatter(y=df['Probabilidad'], x=df['salario en miles'])
# salario alrededor de 0.5
# Valor de x_0 cuando la probabilidad es 0.5 o cerca de 0.5, en mi caso era 0.5 exacto.
a = df['salario en miles'].loc[df['Probabilidad']==0.5]
a = int(a.iloc[0])
plt.axvline(a, linestyle=':', color='green')
plt.axhline(0.5, linestyle='--', color='red')

## Funcion

In [ ]:
# Definir las variables simbólicas
x, x0 = symbols('x x_0')

# Definir la función sigmoide
f = 1 / (1 + exp(-(x - x0)))
f

## Series de Taylor

![Alt Text](taylor_series_formula.png)

In [ ]:
f_prime = diff(f, x)
f_prime_simplified = simplify(f_prime)

f_double_prime = diff(f_prime, x)
f_double_prime_simplified = simplify(f_double_prime)

f_triple_prime = diff(f_double_prime, x)
f_triple_prime_simplified = simplify(f_triple_prime)

# Series de Taylor
f1 = f_prime.subs({x0: a})
f2 = f_double_prime_simplified.subs({x0: a})
f3 = f_triple_prime_simplified.subs({x0: a})

st = f + f1*(x-x0) + f2/2*(x-x0)**2 + f3/6*(x-x0)**3
st

## Integración 
Para calcular el sentimiento I en el intervalo de salarios dado.

#### 1. Método de sumas de Riemann

In [ ]:
def getXRange(xmin, xmax, N, mode):
    dx = (xmin - xmax) / N
    x = np.linspace(xmin, xmax-dx, N)
    if mode.lower() == 'left': 
        x = x
    elif mode.lower() == 'mid': 
        x+-dx/2.0
    else: 
        x = x
    return x

 
def myfunc(x):
    return np.exp(-x)

def myFunc(xmin, xmax):
    return np.exp(-xmin) - np.exp(-xmax)


N = 50
xmin = 0.0
xmax = 5

# create equally spaced x values
xplot=np.linspace(xmin, xmax, np.max(N)* 10)

# get x range
x_left_test = getXRange(xmin, xmax, N, 'left')
x_mid_test = getXRange(xmin, xmax, N, 'mid')
x_right_test = getXRange(xmin, xmax, N, 'right')

# Compute step size
dx = x_left_test[1] - x_left_test[0]
print(f"dx = {dx}")

# plot rieman sum visualization
plt.figure()
plt.plot(xplot, myFunc(xplot), 'r-', label='myFunc')
plt.bar(x_left_test+dx/2, myFunc(x_left_test), dx, alpha=0.5, label='left')
plt.bar(x_mid_test, myFunc(x_mid_test), dx, alpha=0.5, label='mid')
plt.bar(x_right_test-dx/2, myFunc(x_right_test), dx, alpha=0.5, label='right')
plt.legend()
plt.show()

NameError: name 'np' is not defined

In [ ]:
minn_proba = df['Probabilidad'].min()
maxx_proba = df['Probabilidad'].max()
rango_proba = maxx_proba - minn_proba
print(minn_proba, maxx_proba, rango_proba)



#### 2. Método de los trapecios

#### 3. Integrar la función sigmoide
Integrando directamente la función, donde s0 es el punto que escogió para obtener el desarrollo de Taylor.

In [ ]:
# Integrar la función sigmoide
f_a = f.subs({x0: a})
fi = f_a.integrate()
fi

In [ ]:
salarios = list(df['salario en miles'])
reales = []

for salario in salarios:
    reales.append(fi.subs({x: salario}))

df_reales = pd.DataFrame({"valores reales": reales})
df = pd.concat([df, df_reales], axis=1)
df

#### 4. Integrar directamente la aproximación de Taylor

In [ ]:
st_a = st.subs({x0: a})
sti = st_a.integrate()
sti

In [ ]:
salarios = list(df['salario en miles'])
approx = []

for salario in salarios:
    approx.append(sti.subs({x: salario}))

df_aproximados = pd.DataFrame({"valores aproximados": approx})
df = pd.concat([df, df_aproximados], axis=1)
df

### Calcular el valor absoluto de la diferencia entre los valores aproximados y reales

In [ ]:
df['error absoluto'] = abs(df['valores aproximados'] - df['valores reales'])
df.to_csv('output.csv')
df

## Grafique los valores aproximados y los reales y analice el resultado

In [ ]:
maxx = df['error absoluto'].idxmax()
df['salario en miles'].iloc[maxx]

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(df['Probabilidad'], label='valores conocidos', color='green')
ax1.plot(df['error absoluto'], label='error absoluto', linestyle=':',)
# plt.axvline(maxx, linestyle=':', color='green')
# ax1.axhline(0.1, linestyle='--', color='red')
ax1.set_xlabel('Salario en miles')
ax1.set_ylabel('Probabilidad')

ax2 = ax1.twinx()
ax2.plot(df['valores reales'], label='valores reales', linestyle='--', color='blue')
ax2.plot(df['valores aproximados'], label='valores aproximados', linestyle='--', color='orange')
# plt.axhline(0.5, linestyle='--', color='red', label='threshold')
ax2.set_xlabel('salario en miles')
ax2.set_ylabel('Probabilidad')
ax2.legend()